Plug your huggleface token!

In [ ]:
huggingface_token = ""

In [1]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-v1pjjmte
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-v1pjjmte
  Resolved https://github.com/huggingface/transformers.git to commit b9f0fbf532c124ff836466d896a716e26dbe4722
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 66.9 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.6.0.dev0-py3-none-any.whl size=11359424 sha256=db5b2d760955a3b4e8ef3994cf527312d6aa70f2d3610800de363b4fc272d580
  Stored in directory: /tmp/pip-ephem-wheel-cache-dfxg692v/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: hf-xet
    Found existing

In [2]:
import os
import torch
import numpy as np
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    TrainingArguments, 
    Seq2SeqTrainer, 
    GenerationConfig,
    # DataCollatorForLanguageModeling,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# 1. 환경 설정 및 메모리 최적화
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))
model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token

# 3. 4비트 양자화 설정 (QLoRA 핵심)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # T4니까 fp16으로 계산
)

# 4. 모델
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # 4비트로 압축해서 로드
    device_map={"": 0},         # "auto" 대신 0번 GPU로 명시적 고정
    low_cpu_mem_usage=True,     # 로드 시 CPU RAM 부족 방지
    # device_map="auto", # for T4
)

# 5. 양자화 학습 준비 및 LoRA 어댑터 설정 (필수!)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [ ]:
from huggingface_hub import login
login(token=huggingface_token)
# Login using e.g. `huggingface-cli login` to access this dataset
from datasets import load_from_disk, Dataset, load_dataset
ds = load_dataset("Ahmad0067/MedSynth")
len(ds["train"])
# 1. 기본적인 None 값 및 결측치 제거
ds = ds.filter(lambda x: 
    x["Dialogue"] is not None and 
    x[" Note"] is not None and 
    x["ICD10"] is not None
)

# 2. 길이 기반 필터링 (Cleaning)
# 너무 짧은 대화나 노트는 학습에 노이즈가 됩니다. (공백 제외 기준)
min_length = 50 

ds = ds.filter(lambda x: 
    len(str(x["Dialogue"]).strip()) > min_length and 
    len(str(x[" Note"]).strip()) > min_length
)

# 3. 중복 제거 (De-duplication)
# 'Dialogue' 내용이 완전히 똑같은 케이스를 제거하여 데이터 편향을 막습니다.
df = ds["train"].to_pandas()  # 필터링을 위해 잠시 Pandas로 변환
initial_count = len(df)

# 'Dialogue' 컬럼 기준 중복 제거 (첫 번째 것만 남김)
df = df.drop_duplicates(subset=["Dialogue"], keep="first")

print(f"중복 제거 전: {initial_count}개 -> 중복 제거 후: {len(df)}개")

# 다시 Dataset 형태로 복구
ds = Dataset.from_dict(df)

README.md: 0.00B [00:00, ?B/s]

MedSynth_huggingface_final.csv:   0%|          | 0.00/78.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10240 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10240 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10238 [00:00<?, ? examples/s]

중복 제거 전: 10238개 -> 중복 제거 후: 10033개


In [4]:
def tokenize(example):
    dialogue_text = str(example.get("Dialogue", ""))
    # 'Note' 컬럼에 이미 Subjective ~ Plan(Diagnosis 포함)이 다 들어있으므로 그대로 활용
    note_text = str(example.get(" Note", "")) 
    
    # 1. 지시문 (레퍼런스 구조에 맞게 수정)
    instruction = (
        "Task: Convert the medical dialogue into a professional SOAP note.\n"
        "Guidelines:\n"
        "1. STRUCTURE:\n"
        " - Subjective: Patient-reported information (CC, HPI, ROS).\n"
        " - Objective: Observable and measurable findings from physical examination.\n"
        " - Assessment: Clinical diagnosis and medical reasoning.\n"
        " - Plan: Treatment recommendations, prescriptions, and follow-up care.\n"
        "2. CONSTRAINTS:\n"
        " - Use formal clinical terminology.\n"
        " - Use ONLY facts from the dialogue. DO NOT invent names, ages, or data.\n"
        " - Match gender pronouns and anatomical lateralization (Right/Left) strictly.\n"
        " - Start immediately with '**1. Subjective:**'."
    )

    # 2. Prompt 구성
    prompt_header = f"{instruction}\n\nDialogue:\n{dialogue_text}\n\nOutput:\n"
    prompt_ids = tokenizer(
        prompt_header,
        truncation=True,
        max_length=1500, # 대화가 길 수 있으니 넉넉히 할당
        add_special_tokens=False
    )["input_ids"]
    
    # 3. 정답(Answer) 구성
    # 레퍼런스 형식 그대로: Subjective -> Objective -> Assessment(Diagnosis) -> Plan
    full_answer = f"{note_text}"
    
    answer_ids = tokenizer(
        full_answer + tokenizer.eos_token,
        truncation=True,
        max_length=548, # 2048 - 1500 = 548 (여유분)
        add_special_tokens=False
    )["input_ids"]
    
    # 4. 결합 및 Labels 생성 (-100 마스킹)
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids
    attention_mask = [1] * len(input_ids)
    
    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

In [5]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,      # 모델을 넣어주면 패딩 처리를 더 정확히 합니다.
    padding=True      # 가변 길이 데이터를 배치 단위로 예쁘게 맞춰줍니다.
)

# 500개 샘플만 추출
total_count = len(ds)
shuffled_ds = ds.shuffle(seed=42)
small_ds = shuffled_ds.select(range(total_count - 1000))
tokenized_small_ds = small_ds.map(tokenize, remove_columns=small_ds.column_names)

# 'my_tokenized_dataset'이라는 폴더로 저장
# tokenized_small_ds.save_to_disk("/kaggle/working/4000_tokenized_medsynth")

# from datasets import load_from_disk
# tokenized_small_ds = load_from_disk("/kaggle/input/notebooks/ilovesamir/notebookce7024f470/500_tokenized_medsynth")
# # 20개 데이터를 8:2 비율로 나눕니다. (test_size=0.2 -> 4개)
# split_ds = tokenized_small_ds.train_test_split(test_size=0.2, seed=42)

# 1. 먼저 전체에서 10%를 '최종 테스트용(Test)'으로 떼어냅니다.
# 나머지 90%가 임시 학습용 데이터가 됩니다.

# train_ds = split_ds["train"]      # 실제 학습용 (80%)
# eval_ds = split_ds["test"]        # 학습 중 검증용 (10%)
train_ds = tokenized_small_ds.select(range(total_count - 1500))      # 실제 학습용 (80%)
eval_ds = tokenized_small_ds.select(range(total_count - 1500, total_count - 1000))        # 학습 중 검증용 (10%)

print(f"학습(Train) 데이터 개수: {len(train_ds)}")
print(f"검증(Eval) 데이터 개수: {len(eval_ds)}")

Map:   0%|          | 0/9033 [00:00<?, ? examples/s]

학습(Train) 데이터 개수: 8533
검증(Eval) 데이터 개수: 500


In [6]:
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./qwen_7_med",

    # 1. 배치 및 메모리 최적화
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    gradient_checkpointing=True,
    
    # 양자화(QLoRA) 환경에서는 아래 조합이 가장 안정적입니다.
    fp16=True,         # T4 GPU의 필수 옵션
    bf16=False,        # T4는 bf16 미지원

    # 2. 학습 스케줄
    learning_rate=5e-5,
    num_train_epochs=2,
    lr_scheduler_type="cosine",

    # 3. 로그 및 평가
    logging_steps=50, 
    eval_strategy="steps",  # 에폭마다 평가하여 메모리 폭발 방지
    eval_steps=500,

    # [최고 성능 모델 저장을 위한 필수 옵션]
    load_best_model_at_end=True,       # 학습 종료 후 가장 성적이 좋았던 모델을 불러옴
    metric_for_best_model="eval_loss", # 손실(Loss) 값을 기준으로 'Best' 판단
    greater_is_better=False,           # Loss는 낮을수록 좋으므로 False
    
    # 4. 생성 관련
    predict_with_generate=False, 

    # 5. 양자화 전용 최적화 (여기가 중요!)
    optim="paged_adamw_32bit",   # bitsandbytes와 찰떡궁합인 옵티마이저
    max_grad_norm=0.3,           # 수치 안정성을 위해 유지

    # 6. 기타 저장 설정
    save_strategy="steps",             # eval_strategy와 동일하게 맞춤
    save_steps=500,
    save_total_limit=1,                # 용량 관리를 위해 최고점 모델 1개만 보관
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,      # 나눈 학습 데이터
    eval_dataset=eval_ds,        # 나눈 검증 데이터 (여기가 핵심!)
    data_collator=data_collator,
    # compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [7]:
# trainer.train()
trainer.train(resume_from_checkpoint="/kaggle/input/notebooks/robinllee/dial2note-training/qwen_7_med/checkpoint-1500")

Step,Training Loss,Validation Loss
2000,0.454681,0.473948
2134,0.464154,0.473856


TrainOutput(global_step=2134, training_loss=0.13569066361388873, metrics={'train_runtime': 18564.0458, 'train_samples_per_second': 0.919, 'train_steps_per_second': 0.115, 'total_flos': 9.681914528336064e+16, 'train_loss': 0.13569066361388873, 'epoch': 2.0})

In [8]:
# 모델 저장
save_path = "./qwen_7"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("모델 저장 완료!")

모델 저장 완료!
